In [2]:
# 1. Remove any old corrupted files
!rm -rf spark-3.5.0-bin-hadoop3.tgz spark-3.5.0-bin-hadoop3

# 2. Re-download using the stable Archive link
print("Downloading Spark 3.5.0... this may take a minute.")
!wget -q --show-progress https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz

# 3. Extract it
print("Extracting Spark...")
!tar xf spark-3.5.0-bin-hadoop3.tgz

# 4. Final Setup
!pip install -q findspark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

import findspark
findspark.init()
print("Spark is ready!")

spark-3.5.0-bin-had 100%[===================>] 381.85M  9.33MB/s    in 2m 55s  
Extracting Spark...
Spark is ready!


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_extract, when

# Create Spark Session (Fixed with camelCase)
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("LogAnalyzer") \
    .getOrCreate()

print("Spark Session Active!")

Spark Session Active!


In [6]:
import random

# Generating 100,000 log lines to simulate Big Data
def generate_logs(filename, num_lines):
    ips = ["192.168.1.1", "10.0.0.15", "172.16.0.5", "122.11.0.1"]
    methods = ["GET", "POST", "DELETE"]
    endpoints = ["/login", "/api/data", "/admin/config", "/home"]
    user_agents = ["Mozilla/5.0", "Python-requests/2.25", "Nmap-Scanner"]

    with open(filename, "w") as f:
        for _ in range(num_lines):
            ip = random.choice(ips)
            method = random.choice(methods)
            endpoint = random.choice(endpoints)
            # Injecting SQL Injection threats for analysis
            if random.random() < 0.05:
                endpoint = "/admin/config?query=DROP+TABLE+users"

            status = random.choice([200, 401, 404, 500])
            f.write(f'{ip} - - [15/Mar/2026:10:00:00 +0000] "{method} {endpoint} HTTP/1.1" {status} 512 "{random.choice(user_agents)}"\n')

generate_logs("server_logs.txt", 100000)
print("Log file 'server_logs.txt' is ready for distributed analysis!")

Log file 'server_logs.txt' is ready for distributed analysis!


In [7]:
from pyspark.sql.functions import regexp_extract, col

# 1. Read the raw text file into Spark
raw_logs_df = spark.read.text("server_logs.txt")

# 2. Define the Regex pattern for the Apache log format
# This pattern groups the IP, the Method, the Endpoint, the Status, and the User Agent
log_pattern = r'^(\S+) \S+ \S+ \[(.*?)\] "(.*?) (.*?) \S+" (\d+) (\d+) "(.*?)"'

# 3. Apply the pattern to create columns
logs_df = raw_logs_df.select(
    regexp_extract('value', log_pattern, 1).alias('ip'),
    regexp_extract('value', log_pattern, 4).alias('endpoint'),
    regexp_extract('value', log_pattern, 5).cast('integer').alias('status'),
    regexp_extract('value', log_pattern, 7).alias('user_agent')
)

# Show the structured data
print("Structured Log Table:")
logs_df.show(10, truncate=False)

Structured Log Table:
+-----------+-------------+------+--------------------+
|ip         |endpoint     |status|user_agent          |
+-----------+-------------+------+--------------------+
|122.11.0.1 |/admin/config|500   |Mozilla/5.0         |
|10.0.0.15  |/admin/config|500   |Python-requests/2.25|
|122.11.0.1 |/admin/config|401   |Mozilla/5.0         |
|10.0.0.15  |/api/data    |404   |Mozilla/5.0         |
|192.168.1.1|/api/data    |404   |Python-requests/2.25|
|172.16.0.5 |/admin/config|401   |Python-requests/2.25|
|10.0.0.15  |/home        |500   |Nmap-Scanner        |
|192.168.1.1|/admin/config|404   |Nmap-Scanner        |
|172.16.0.5 |/home        |500   |Nmap-Scanner        |
|10.0.0.15  |/login       |500   |Nmap-Scanner        |
+-----------+-------------+------+--------------------+
only showing top 10 rows



In [8]:
from pyspark.sql.functions import when

# Define threat logic
analyzed_df = logs_df.withColumn("threat_type",
    when(col("endpoint").contains("DROP") | col("endpoint").contains("SELECT"), "SQL_INJECTION")
    .when(col("status") == 401, "UNAUTHORIZED_ACCESS_ATTEMPT")
    .when(col("user_agent").contains("Nmap"), "NETWORK_SCANNER_DETECTED")
    .otherwise("SAFE")
)

# Filter to see only the threats
print("Detected Security Threats:")
analyzed_df.filter(col("threat_type") != "SAFE").show(10)

Detected Security Threats:
+-----------+-------------+------+--------------------+--------------------+
|         ip|     endpoint|status|          user_agent|         threat_type|
+-----------+-------------+------+--------------------+--------------------+
| 122.11.0.1|/admin/config|   401|         Mozilla/5.0|UNAUTHORIZED_ACCE...|
| 172.16.0.5|/admin/config|   401|Python-requests/2.25|UNAUTHORIZED_ACCE...|
|  10.0.0.15|        /home|   500|        Nmap-Scanner|NETWORK_SCANNER_D...|
|192.168.1.1|/admin/config|   404|        Nmap-Scanner|NETWORK_SCANNER_D...|
| 172.16.0.5|        /home|   500|        Nmap-Scanner|NETWORK_SCANNER_D...|
|  10.0.0.15|       /login|   500|        Nmap-Scanner|NETWORK_SCANNER_D...|
|  10.0.0.15|/admin/config|   401|Python-requests/2.25|UNAUTHORIZED_ACCE...|
|192.168.1.1|    /api/data|   401|Python-requests/2.25|UNAUTHORIZED_ACCE...|
|192.168.1.1|/admin/config|   500|        Nmap-Scanner|NETWORK_SCANNER_D...|
| 122.11.0.1|       /login|   401|        Nmap-Sc

In [9]:
# Aggregate the results
threat_summary = analyzed_df.groupBy("threat_type").count().orderBy("count", ascending=False)
threat_summary.show()

+--------------------+-----+
|         threat_type|count|
+--------------------+-----+
|                SAFE|47546|
|NETWORK_SCANNER_D...|23740|
|UNAUTHORIZED_ACCE...|23730|
|       SQL_INJECTION| 4984|
+--------------------+-----+



In [10]:
# Create a directory for the results
output_path = "threat_reports_parquet"

# Save the analyzed threats as Parquet
# 'overwrite' ensures you can run this cell multiple times without errors
analyzed_df.write.mode("overwrite").parquet(output_path)

print(f"Success! Threat reports saved to: {output_path}")

# Verify by reading it back
parquet_data = spark.read.parquet(output_path)
print(f"Verified: Read {parquet_data.count()} rows from Parquet file.")

Success! Threat reports saved to: threat_reports_parquet
Verified: Read 100000 rows from Parquet file.
